# Notebook 1: Base Stack Export

Exports multi-band rasters at MODIS scale in standard geographic coordinates (WGS84) per basin.
To completely bypass GEE's **User memory limit exceeded** errors, we split the 34-band composite into **three modular, lightweight assets** exported in parallel:

1. **`NppStack` (24 bands)**: MODIS NPP median + 23 annual bands. Already at MODIS scale (zero `reduceResolution` memory overhead).
2. **`GediStack` (3 bands)**: GEDI L2B UOI, N count, and L2A rh98 height (only 3 `reduceResolution` chains).
3. **`CovStack` (7 bands)**: Flood frequency, forest fraction, elevation, slope, hnd, precip, clay (only 7 `reduceResolution` chains).

### Running downstream:
Stage 2 loads these three static assets and concatenates them in one millisecond via `ee.Image.cat([npp, gedi, covs])`, running downstream calculations with **zero** live reprojection memory overhead!

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Output destination
GEE_PROJECT = 'quantum-bonus-434714-t2'
ASSET_ROOT = f'projects/{GEE_PROJECT}/assets/DefaunationFromSpace'

# Study Regions — per-basin exports avoid spanning the Atlantic
CONGO_BOUNDS = [8, -12, 35, 8]
AMAZON_BOUNDS = [-73, -18, -44, 8]
CONGO_BBOX = ee.Geometry.Rectangle(CONGO_BOUNDS)
AMAZON_BBOX = ee.Geometry.Rectangle(AMAZON_BOUNDS)
BASINS = [('Congo', CONGO_BBOX, CONGO_BOUNDS), ('Amazon', AMAZON_BBOX, AMAZON_BOUNDS)]

# Spatial grid for GEDI exports (reduces per-task memory pressure)
GEDI_GRID_COLS = 3
GEDI_GRID_ROWS = 3

# Years
YEARS = list(range(2001, 2024))  # 2001-2023 inclusive

# GEE Dataset IDs
MODIS_NPP     = 'MODIS/061/MOD17A3HGF'
GLOFAS        = 'JRC/CEMS_GLOFAS/FloodHazard/v2_1'
MERIT_HYDRO   = 'MERIT/Hydro/v1_0_1'
FOREST_MASK   = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS  = 10   # Undisturbed since ~1982
SRTM          = 'USGS/SRTMGL1_003'
GEDI_L2B      = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
GEDI_L2A      = 'LARSE/GEDI/GEDI02_A_002_MONTHLY'
CHIRPS        = 'UCSB-CHG/CHIRPS/DAILY'
SOILGRIDS_CLAY = 'OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02'

# GEDI date range
GEDI_START = '2020-01-01'
GEDI_END   = '2023-12-31'

# CHIRPS date range (climatological mean)
PRECIP_START = '2001-01-01'
PRECIP_END   = '2023-12-31'

# --- MODIS NPP Reference Projection ---
_modis_col = ee.ImageCollection(MODIS_NPP).select('Npp')
MODIS_PROJ = ee.Image(_modis_col.first()).projection()
MODIS_SCALE = 463.3127165279165  # MODIS equatorial pixel size in meters

print("\u2713 Configuration loaded.")
print(f"  Basins: {[b[0] for b in BASINS]}")
print(f"  GEDI grid: {GEDI_GRID_COLS}x{GEDI_GRID_ROWS} = {GEDI_GRID_COLS * GEDI_GRID_ROWS} tiles per band per basin")
print(f"  Asset root: {ASSET_ROOT}")

In [ ]:
# =============================================================================
# BLOCK 2: BAND-BUILDING FUNCTIONS
# =============================================================================

def build_npp_bands(basin_geom):
    modis = ee.ImageCollection(MODIS_NPP).select('Npp').filterBounds(basin_geom)
    
    def get_annual(year):
        year = ee.Number(year)
        return modis.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).first().clip(basin_geom).set('year', year)
    
    annual_imgs = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(get_annual)
    )
    
    median_npp = annual_imgs.median().clip(basin_geom).rename('Npp_median')
    annual_npp = annual_imgs.toBands().clip(basin_geom)
    band_names = [f'NPP_{y}' for y in YEARS]
    annual_npp = annual_npp.rename(band_names)
    
    return median_npp, annual_npp

def build_flood_frequency(basin_geom):
    glofas_col = ee.ImageCollection(GLOFAS).filterBounds(basin_geom)
    glofas = glofas_col.mosaic().clip(basin_geom)
    depth_bands = ['RP10_depth', 'RP20_depth', 'RP50_depth', 'RP75_depth',
                   'RP100_depth', 'RP200_depth', 'RP500_depth']
    
    flood_freq = glofas.select(depth_bands).gte(0).reduce(ee.Reducer.sum()).unmask()
    
    flood_proj = ee.Image(glofas_col.first()).projection()
    hnd_mask = ee.Image(MERIT_HYDRO).select('hnd').gt(0).clip(basin_geom)
    flood_freq = flood_freq.setDefaultProjection(crs=flood_proj).updateMask(hnd_mask)
    
    flood_reduced = flood_freq.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('flood_freq')
    
    return flood_reduced

def build_forest_fraction(basin_geom):
    tmf_col = ee.ImageCollection(FOREST_MASK).filterBounds(basin_geom)
    tmf_proj = tmf_col.first().projection()
    
    forest = tmf_col.mosaic().eq(FOREST_CLASS).clip(basin_geom).setDefaultProjection(tmf_proj)
    
    forest_frac = forest.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('forest_fraction')
    
    return forest_frac

def build_terrain(basin_geom):
    srtm = ee.Image(SRTM).clip(basin_geom)
    srtm_proj = srtm.select('elevation').projection()
    
    elev = srtm.select('elevation').setDefaultProjection(srtm_proj)
    elev_reduced = elev.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('elevation')
    
    slp = ee.Terrain.slope(srtm).clip(basin_geom).setDefaultProjection(srtm_proj)
    slp_reduced = slp.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('slope')
    
    return elev_reduced, slp_reduced

def build_hnd(basin_geom):
    hnd = ee.Image(MERIT_HYDRO).select('hnd').clip(basin_geom)
    hnd_proj = hnd.projection()
    
    hnd_reduced = hnd.setDefaultProjection(hnd_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('hnd')
    
    return hnd_reduced

def make_grid(bounds, n_cols, n_rows):
    """Split a bounding box [lon_min, lat_min, lon_max, lat_max] into a grid of ee.Geometry.Rectangles."""
    lon_min, lat_min, lon_max, lat_max = bounds
    lon_step = (lon_max - lon_min) / n_cols
    lat_step = (lat_max - lat_min) / n_rows
    cells = []
    for r in range(n_rows):
        for c in range(n_cols):
            cells.append(ee.Geometry.Rectangle([
                lon_min + c * lon_step,
                lat_min + r * lat_step,
                lon_min + (c + 1) * lon_step,
                lat_min + (r + 1) * lat_step
            ]))
    return cells

def build_gedi(basin_geom):
    # Select only the bands we need to minimize memory footprint
    gedi_b = ee.ImageCollection(GEDI_L2B).filterBounds(basin_geom).filterDate(GEDI_START, GEDI_END).select(['pai', 'pavd_z0'])
    
    # Calculate GEDI_UOI using a pipeline (map-then-reduce) pattern:
    # Compute UOI = (pai - pavd) / pai per image, then take temporal mean at native 25m resolution.
    def calc_uoi(img):
        pai = img.select('pai')
        pavd = img.select('pavd_z0')
        return pai.subtract(pavd).divide(pai).clamp(0, 1).updateMask(pai.gt(0)).rename('GEDI_UOI')
    
    uoi_mean = gedi_b.map(calc_uoi).mean().setDefaultProjection(crs='EPSG:4326', scale=25)
    uoi_count = gedi_b.select('pai').count().rename('GEDI_N').setDefaultProjection(crs='EPSG:4326', scale=25)
    
    gedi_a = ee.ImageCollection(GEDI_L2A).filterBounds(basin_geom).filterDate(GEDI_START, GEDI_END)
    rh98_mean = gedi_a.select('rh98').mean().rename('GEDI_rh98').setDefaultProjection(crs='EPSG:4326', scale=25)
    
    return uoi_mean, uoi_count, rh98_mean

def build_precip(basin_geom):
    chirps = ee.ImageCollection(CHIRPS).filterBounds(basin_geom).filterDate(PRECIP_START, PRECIP_END)
    chirps_proj = chirps.first().projection()
    
    def annual_total(year):
        year = ee.Number(year)
        return chirps.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).sum().set('year', year)
    
    annual_precip = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(annual_total)
    )
    mean_precip = annual_precip.mean().clip(basin_geom).rename('precip').setDefaultProjection(chirps_proj)
    
    return mean_precip

def build_clay(basin_geom):
    clay = ee.Image(SOILGRIDS_CLAY).clip(basin_geom)
    clay_mean = clay.reduce(ee.Reducer.mean()).rename('clay')
    clay_proj = clay.select(0).projection()
    
    clay_reduced = clay_mean.setDefaultProjection(clay_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    )
    
    return clay_reduced


def build_gedi_quality_mask(basin_geom):
    """Builds a 2-band quality mask image at 25m from the raw GEDI L2B collection.
    
    Since 97% of GEDI pixels have exactly 1 shot (N=1), quality filtering
    post-temporal-average is functionally equivalent to pre-average filtering.
    We use temporal min/max to conservatively flag any pixel where ANY
    contributing shot was low-quality or had degraded pointing.
    
    L2B flags cover both UOI (L2B-derived) and rh98 (L2A-derived) because
    both products originate from the same physical shot.
    
    Returns:
        ee.Image with 2 bands:
          - quality_min  : min(l2b_quality_flag) across months. 1 = all shots good.
          - degrade_max  : max(degrade_flag) across months. 0 = no shots degraded.
    """
    gedi_b = (ee.ImageCollection(GEDI_L2B)
              .filterBounds(basin_geom)
              .filterDate(GEDI_START, GEDI_END))
    
    quality_min = (gedi_b.select('l2b_quality_flag')
                   .min()
                   .setDefaultProjection(crs='EPSG:4326', scale=25)
                   .rename('quality_min'))
    
    degrade_max = (gedi_b.select('degrade_flag')
                   .max()
                   .setDefaultProjection(crs='EPSG:4326', scale=25)
                   .rename('degrade_max'))
    
    return ee.Image.cat([quality_min, degrade_max]).toUint8()

print("\u2713 All band functions defined.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running unit tests (using Congo BBox)...\n")
    passed = 0
    failed = 0
    
    # --- Test 1: make_grid cell count and bounds ---
    try:
        print("  [1/7] Validating make_grid()...")
        grid = make_grid(CONGO_BOUNDS, GEDI_GRID_COLS, GEDI_GRID_ROWS)
        expected = GEDI_GRID_COLS * GEDI_GRID_ROWS
        assert len(grid) == expected, f"Expected {expected} cells, got {len(grid)}"
        
        # Verify first and last cell bounds reconstruct the original bbox
        first = grid[0].bounds().coordinates().getInfo()[0]
        last = grid[-1].bounds().coordinates().getInfo()[0]
        lon_min_actual = min(c[0] for c in first)
        lat_min_actual = min(c[1] for c in first)
        lon_max_actual = max(c[0] for c in last)
        lat_max_actual = max(c[1] for c in last)
        assert abs(lon_min_actual - CONGO_BOUNDS[0]) < 0.1, f"lon_min mismatch: {lon_min_actual}"
        assert abs(lat_min_actual - CONGO_BOUNDS[1]) < 0.1, f"lat_min mismatch: {lat_min_actual}"
        assert abs(lon_max_actual - CONGO_BOUNDS[2]) < 0.1, f"lon_max mismatch: {lon_max_actual}"
        assert abs(lat_max_actual - CONGO_BOUNDS[3]) < 0.1, f"lat_max mismatch: {lat_max_actual}"
        
        passed += 1
        print(f"    \u2713 make_grid: {expected} cells, bounds match original bbox")
    except Exception as e:
        failed += 1
        print(f"    \u2717 make_grid FAILED: {e}")
        
    # --- Test 2: NPP Stack band count and names ---
    try:
        print("  [2/7] Validating NPP Stack (24 bands)...")
        npp_med, npp_ann = build_npp_bands(CONGO_BBOX)
        npp_stack = ee.Image.cat([npp_med, npp_ann]).toFloat()
        band_names = npp_stack.bandNames().getInfo()
        assert len(band_names) == 24, f"Expected 24 bands, got {len(band_names)}"
        assert band_names[0] == 'Npp_median', f"First band should be 'Npp_median', got '{band_names[0]}'"
        expected_annual = [f'NPP_{y}' for y in YEARS]
        assert band_names[1:] == expected_annual, f"Annual band names mismatch"
        passed += 1
        print(f"    \u2713 NPP Stack: {len(band_names)} bands, names verified")
    except Exception as e:
        failed += 1
        print(f"    \u2717 NPP Stack FAILED: {e}")
        
    # --- Test 3: GEDI band names ---
    try:
        print("  [3/7] Validating GEDI bands (3 individual bands)...")
        uoi, n, rh98 = build_gedi(CONGO_BBOX)
        uoi_name = uoi.bandNames().getInfo()
        n_name = n.bandNames().getInfo()
        rh98_name = rh98.bandNames().getInfo()
        assert uoi_name == ['GEDI_UOI'], f"UOI band name: expected ['GEDI_UOI'], got {uoi_name}"
        assert n_name == ['GEDI_N'], f"N band name: expected ['GEDI_N'], got {n_name}"
        assert rh98_name == ['GEDI_rh98'], f"rh98 band name: expected ['GEDI_rh98'], got {rh98_name}"
        passed += 1
        print("    \u2713 GEDI bands: GEDI_UOI, GEDI_N, GEDI_rh98 verified")
    except Exception as e:
        failed += 1
        print(f"    \u2717 GEDI bands FAILED: {e}")
    
    # --- Test 4: UOI value range sanity check ---
    try:
        print("  [4/7] Validating UOI value range (sample point)...")
        # Sample a forested point in central Congo (~20E, 0N)
        test_point = ee.Geometry.Point([20.0, 0.0])
        uoi, _, _ = build_gedi(test_point.buffer(25000))
        sample = uoi.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=test_point.buffer(10000),
            scale=463,
            tileScale=4
        ).getInfo()
        uoi_val = sample.get('GEDI_UOI')
        if uoi_val is None:
            print("    \u26A0 UOI sample is None (no GEDI data at test point) — skipping range check")
            passed += 1
        else:
            assert 0 <= uoi_val <= 1, f"UOI out of range [0,1]: {uoi_val}"
            passed += 1
            print(f"    \u2713 UOI value at test point: {uoi_val:.4f} (in [0, 1])")
    except Exception as e:
        failed += 1
        print(f"    \u2717 UOI value range FAILED: {e}")
    
    # --- Test 5: Covariates band count and names ---
    try:
        print("  [5/7] Validating Covariates Stack (7 bands)...")
        flood = build_flood_frequency(CONGO_BBOX)
        forest = build_forest_fraction(CONGO_BBOX)
        elev, slope = build_terrain(CONGO_BBOX)
        hnd = build_hnd(CONGO_BBOX)
        precip = build_precip(CONGO_BBOX)
        clay = build_clay(CONGO_BBOX)
        cov_stack = ee.Image.cat([flood, forest, elev, slope, hnd, precip, clay]).toFloat()
        band_names = cov_stack.bandNames().getInfo()
        expected_names = ['flood_freq', 'forest_fraction', 'elevation', 'slope', 'hnd', 'precip', 'clay']
        assert band_names == expected_names, f"Cov band names mismatch: {band_names} != {expected_names}"
        passed += 1
        print(f"    \u2713 Covariates Stack: {len(band_names)} bands, names verified")
    except Exception as e:
        failed += 1
        print(f"    \u2717 Covariates Stack FAILED: {e}")
    
    # --- Test 6: Full assembly band count (32 total) ---
    try:
        print("  [6/7] Validating full assembly (32 bands)...")
        npp_med, npp_ann = build_npp_bands(CONGO_BBOX)
        npp_stack = ee.Image.cat([npp_med, npp_ann]).toFloat()
        uoi, n, rh98 = build_gedi(CONGO_BBOX)
        flood = build_flood_frequency(CONGO_BBOX)
        forest = build_forest_fraction(CONGO_BBOX)
        elev, slope = build_terrain(CONGO_BBOX)
        hnd = build_hnd(CONGO_BBOX)
        precip = build_precip(CONGO_BBOX)
        clay = build_clay(CONGO_BBOX)
        cov_stack = ee.Image.cat([flood, forest, elev, slope, hnd, precip, clay]).toFloat()
        full = ee.Image.cat([npp_stack, uoi, n, rh98, cov_stack]).toFloat()
        n_bands = len(full.bandNames().getInfo())
        assert n_bands == 34, f"Expected 34 bands (24 NPP + 3 GEDI + 7 Cov), got {n_bands}"
        passed += 1
        print(f"    \u2713 Full assembly: {n_bands} bands")
    except Exception as e:
        failed += 1
        print(f"    \u2717 Full assembly FAILED: {e}")
        

    # --- Test 7: build_gedi_quality_mask band names ---
    try:
        print("  [7/7] Validating build_gedi_quality_mask()...")
        qmask = build_gedi_quality_mask(CONGO_BBOX)
        assert isinstance(qmask, ee.Image), "Quality mask must return an ee.Image"
        qmask_bands = qmask.bandNames().getInfo()
        assert qmask_bands == ['quality_min', 'degrade_max'], f"Unexpected bands: {qmask_bands}"
        passed += 1
        print(f"    \u2713 Quality mask: {qmask_bands} verified")
    except Exception as e:
        failed += 1
        print(f"    \u2717 build_gedi_quality_mask FAILED: {e}")
    total = passed + failed
    print(f"\n{'='*60}")
    if failed == 0:
        print(f"  \u2713 ALL {passed}/{total} TESTS PASSED")
    else:
        print(f"  \u2717 {passed}/{total} passed, {failed} failed. Fix failures before proceeding.")
    print(f"{'='*60}")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: MODULAR PARALLEL EXPORTS
# =============================================================================

def safe_start(task, asset_id):
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_modular_stacks(dry_run=True):
    """Launches exports for modular assets per basin to GEE Assets.
    
    Per basin:
    1. NPP Stack (24 bands - MODIS scale, 0 reduceResolution overhead)
    2-4. GEDI UOI/N/Rh98 (1 band each, spatially gridded into GEDI_GRID_COLS x GEDI_GRID_ROWS tiles)
    5. Cov Stack (7 bands - environmental grid reduction)
    """
    tasks = []
    n_gedi_tiles = GEDI_GRID_COLS * GEDI_GRID_ROWS
    
    for basin_name, basin_geom, basin_bounds in BASINS:
        # --- NPP STACK ---
        npp_med, npp_ann = build_npp_bands(basin_geom)
        npp_stack = ee.Image.cat([npp_med, npp_ann]).toFloat()
        npp_id = f'{ASSET_ROOT}/NppStack_{basin_name}'
        
        task_npp = ee.batch.Export.image.toAsset(
            image=npp_stack,
            description=f'NppStack_{basin_name}',
            assetId=npp_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task_npp, npp_id))
        
        # --- GEDI STACKS (spatially gridded to native 25m resolution) ---
        grid = make_grid(basin_bounds, GEDI_GRID_COLS, GEDI_GRID_ROWS)
        
        for gi, cell_geom in enumerate(grid):
            uoi, n, rh98 = build_gedi(cell_geom)
            
            gedi_stack = ee.Image.cat([uoi, n, rh98]).toFloat()
            gedi_id = f'{ASSET_ROOT}/GediStack_{basin_name}_{gi}'
            task_gedi = ee.batch.Export.image.toAsset(
                image=gedi_stack,
                description=f'GediStack_{basin_name}_{gi}',
                assetId=gedi_id,
                region=cell_geom,
                scale=25,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((task_gedi, gedi_id))
        
        # --- COVARIATES STACK ---
        flood = build_flood_frequency(basin_geom)
        forest = build_forest_fraction(basin_geom)
        elev, slope = build_terrain(basin_geom)
        hnd = build_hnd(basin_geom)
        precip = build_precip(basin_geom)
        clay = build_clay(basin_geom)
        
        cov_stack = ee.Image.cat([
            flood, forest, elev, slope, hnd, precip, clay
        ]).toFloat()
        cov_id = f'{ASSET_ROOT}/CovStack_{basin_name}'
        
        task_cov = ee.batch.Export.image.toAsset(
            image=cov_stack,
            description=f'CovStack_{basin_name}',
            assetId=cov_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task_cov, cov_id))
        
    n_per_basin = 2 + n_gedi_tiles  # NPP + CovStack + 3 GEDI bands * grid tiles
    print(f"\u2713 {len(tasks)} parallel tasks configured ({n_per_basin} per basin):")
    for _, aid in tasks:
        print(f"    {aid}")
        
    if dry_run:
        print("\nDRY RUN. Call export_modular_stacks(dry_run=False) to launch.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
            print(f"  \u2713 Started modular task: {asset_id.split('/')[-1]}")
        print(f"\n\u2713 All {len(tasks)} parallel tasks started successfully!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")

export_modular_stacks(dry_run=True)

In [ ]:
# =============================================================================
# BLOCK 5: GEDI QUALITY MASK EXPORT
#
# Exports the quality mask images built by build_gedi_quality_mask() (Block 2)
# as static 25m assets. These are loaded in NB2 and applied before
# reduceResolution, avoiding recomputation of the raw GEDI collections.
# =============================================================================

def export_gedi_quality_masks(dry_run=True):
    """Exports GEDI quality masks per basin (full-basin, no tiling needed)."""
    tasks = []
    
    for basin_name, basin_geom, basin_bounds in BASINS:
        qmask = build_gedi_quality_mask(basin_geom)
        asset_id = f'{ASSET_ROOT}/GediQuality_{basin_name}'
        
        task = ee.batch.Export.image.toAsset(
            image=qmask,
            description=f'GediQuality_{basin_name}',
            assetId=asset_id,
            region=basin_geom,
            scale=25,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task, asset_id))
    
    print(f'\u2713 {len(tasks)} quality mask export tasks configured:')
    for _, aid in tasks:
        print(f'    {aid}')
    
    if dry_run:
        print('\nDRY RUN. Call export_gedi_quality_masks(dry_run=False) to launch.')
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
            print(f'  \u2713 Started: {asset_id.split("/")[-1]}')
        print('\n\u2713 All quality mask exports started.')
        print('  Monitor at: https://code.earthengine.google.com/tasks')

export_gedi_quality_masks(dry_run=True)
